# Hipótesis 1

 Factores de entorno: La iluminación (P43_1B), la vigilancia en zonas de obra (P43_1G) y la señalización (P43_1A) son los factores más asociados con la percepción de empeoramiento de seguridad.

In [45]:
# =============================================================================
# 0. CONFIGURACIÓN
# =============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import Logit
from statsmodels.miscmodels.ordinal_model import OrderedModel
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
pd.set_option('display.max_columns', 50)

DATA_DIR = './datos_limpios'
corredores = pd.read_csv(f'{DATA_DIR}/corredores_limpio.csv')

# 1. Selección de Variables

In [46]:
# =============================================================================
# 1. SELECCIÓN DE VARIABLES
# =============================================================================
cols_p43_1 = [c for c in corredores.columns if c.startswith('P43_1')]

vars_h3 = ['seguridad_empeoro'] + cols_p43_1 + ['obras_iniciadas', 'sector', 'tamano_empresa', 'tramo', 'P1', 'indice_satisfaccion']

df = corredores[vars_h3].copy()

# Nombres descriptivos (para visualizaciones y reportes)
labels = {
    'seguridad_empeoro'  : 'Percepción de seguridad empeoró',
    'P43_1A'           : 'Señalización',
    'P43_1B'           : 'Iluminación',
    'P43_1C'           : 'Manejo de Basuras',
    'P43_1D'           : 'Reubicación de estación SITP o Transmilenio',
    'P43_1E'           : 'Espacios adecuados para transitar',
    'P43_1F'           : 'Presencia de habitantes de calle',
    'P43_1G'           : 'Vigilancia en zonas de obra',
    'obras_iniciadas'  : 'Obras iniciadas en el tramo',
    'sector'           : 'Sector productivo',
    'tamano_empresa'   : 'Tamaño empresa',
    'tramo'            : 'Tramo geográfico',
    'P1'               : 'Número de empleados',
    'indice_satisfaccion': 'Índice de satisfacción con el entorno',
}

print(f"Dimensiones: {df.shape}")
print(f"\nVariables y descripciones:")
for col in df.columns:
    print(f"  {col:<20} → {labels.get(col, '—')}")
df.head()

Dimensiones: (2105, 14)

Variables y descripciones:
  seguridad_empeoro    → Percepción de seguridad empeoró
  P43_1A               → Señalización
  P43_1B               → Iluminación
  P43_1C               → Manejo de Basuras
  P43_1D               → Reubicación de estación SITP o Transmilenio
  P43_1E               → Espacios adecuados para transitar
  P43_1F               → Presencia de habitantes de calle
  P43_1G               → Vigilancia en zonas de obra
  obras_iniciadas      → Obras iniciadas en el tramo
  sector               → Sector productivo
  tamano_empresa       → Tamaño empresa
  tramo                → Tramo geográfico
  P1                   → Número de empleados
  indice_satisfaccion  → Índice de satisfacción con el entorno


,seguridad_empeoro,P43_1A,P43_1B,P43_1C,P43_1D,P43_1E,P43_1F,P43_1G,obras_iniciadas,sector,tamano_empresa,tramo,P1,indice_satisfaccion
0,1,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0,Servicio,Grande,18,3.0,3.250000
1,0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0,NaN,Pequeña,18,1.0,1.500000
2,0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0,Servicio,Grande,18,1.0,1.000000
3,0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0,Comercio,Micro,18,5.0,2.833333
4,0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,0,Servicio,Micro,18,4.0,2.333333


# 2. One Hot Encoding

Realizamos el proceso a las categóricas: sector, tamano_empresa y tramo

In [47]:
# =============================================================================
# 2. ONE HOT ENCODING
# =============================================================================

df_encoded = df.copy()

# --- sector: OHE eliminando 'Comercio' como categoría de referencia ---
df_encoded['sector'] = df_encoded['sector'].replace('NS/NR', np.nan)
ohe_sector = pd.get_dummies(df_encoded['sector'], prefix='sector', dummy_na=False)
ohe_sector = ohe_sector.drop(columns=['sector_Comercio'], errors='ignore')

# --- tamano_empresa: OHE eliminando 'Micro' como categoría de referencia ---
df_encoded['tamano_empresa'] = df_encoded['tamano_empresa'].replace('NS/NR', np.nan)
ohe_tamano = pd.get_dummies(df_encoded['tamano_empresa'], prefix='tamano_empresa', dummy_na=False)
ohe_tamano = ohe_tamano.drop(columns=['tamano_empresa_Micro'], errors='ignore')

# --- tramo: OHE eliminando tramo 1 como categoría de referencia ---
ohe_tramo = pd.get_dummies(df_encoded['tramo'], prefix='tramo', dummy_na=False)
ohe_tramo = ohe_tramo.drop(columns=['tramo_1'], errors='ignore')

# Reemplazar columnas originales por sus dummies
df_encoded = pd.concat([
    df_encoded.drop(columns=['sector', 'tamano_empresa', 'tramo']),
    ohe_sector,
    ohe_tamano,
    ohe_tramo
], axis=1)

print("One Hot Encoding completado — k-1 variables por categoría:")
print(f"  sector        : {list(ohe_sector.columns)}")
print(f"  tamano_empresa: {list(ohe_tamano.columns)}")
print(f"  tramo         : {list(ohe_tramo.columns[:5])} ... ({ohe_tramo.shape[1]} dummies)")
print(f"\nDimensiones finales: {df_encoded.shape}")
df_encoded.head()

One Hot Encoding completado — k-1 variables por categoría:
  sector        : ['sector_Industria', 'sector_Servicio']
  tamano_empresa: ['tamano_empresa_Grande', 'tamano_empresa_Mediana', 'tamano_empresa_Pequeña']
  tramo         : ['tramo_2', 'tramo_3', 'tramo_4', 'tramo_5', 'tramo_6'] ... (18 dummies)

Dimensiones finales: (2105, 34)


,seguridad_empeoro,P43_1A,P43_1B,P43_1C,P43_1D,P43_1E,P43_1F,P43_1G,obras_iniciadas,P1,indice_satisfaccion,sector_Industria,sector_Servicio,tamano_empresa_Grande,tamano_empresa_Mediana,tamano_empresa_Pequeña,tramo_2,tramo_3,tramo_4,tramo_5,tramo_6,tramo_7,tramo_8,tramo_9,tramo_10,tramo_11,tramo_12,tramo_13,tramo_14,tramo_15,tramo_16,tramo_17,tramo_18,tramo_19
0,1,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0,3.0,3.250000,False,True,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
1,0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0,1.0,1.500000,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
2,0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0,1.0,1.000000,False,True,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
3,0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0,5.0,2.833333,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
4,0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,0,4.0,2.333333,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False


# 3. Manejo de Valores Nulos

In [48]:
# =============================================================================
# 3. REPORTE DE MISSINGS
# =============================================================================

missings = df_encoded.isnull().sum()
missings = missings[missings > 0].sort_values(ascending=False)

total_filas = len(df_encoded)
filas_con_missing = df_encoded.isnull().any(axis=1).sum()

reporte = pd.DataFrame({
    'Missings':    missings,
    'Porcentaje':  (missings / total_filas * 100).round(2)
})

print(f"Total filas: {total_filas}")
print(f"Filas con al menos un missing: {filas_con_missing} ({filas_con_missing / total_filas * 100:.2f}%)")
print(f"Columnas con missings: {len(reporte)} de {df_encoded.shape[1]}\n")
print(reporte.to_string())

Total filas: 2105
Filas con al menos un missing: 84 (3.99%)
Columnas con missings: 8 de 34

        Missings  Porcentaje
P43_1A        62        2.95
P43_1B        62        2.95
P43_1C        62        2.95
P43_1D        62        2.95
P43_1E        62        2.95
P43_1F        62        2.95
P43_1G        62        2.95
P1            23        1.09


In [49]:
# Eliminamos los missings
df_encoded = df_encoded.dropna()
print(f"Filas eliminadas: {2105 - len(df_encoded)}")
print(f"Filas restantes:  {len(df_encoded)}")

Filas eliminadas: 84
Filas restantes:  2021


# 4. Regresión Logística

In [50]:
# =============================================================================
# 4. REGRESIÓN LOGÍSTICA — INFERENCIA CAUSAL (sin variables de tramo)
# =============================================================================
from statsmodels.stats.outliers_influence import variance_inflation_factor

# --- Separar variable dependiente y regresores ---
y = df_encoded['seguridad_empeoro']
X = df_encoded.drop(columns=['seguridad_empeoro'])

# Convertir bool → int (dummies OHE)
X = X.astype(int)

# Eliminar dummies de tramo
cols_tramo = [c for c in X.columns if c.startswith('tramo_')]
X1 = X.drop(columns=cols_tramo)

# Agregar constante
X1_const = sm.add_constant(X1)

# --- Estimación Logit con errores robustos HC1 ---
modelo_logit = Logit(y, X1_const).fit(cov_type='HC1', disp=False)
print(modelo_logit.summary())

# --- Odds Ratios con IC 95% ---
or_tabla = pd.DataFrame({
    'Odds Ratio': np.exp(modelo_logit.params),
    'IC 2.5%':    np.exp(modelo_logit.conf_int()[0]),
    'IC 97.5%':   np.exp(modelo_logit.conf_int()[1]),
    'p-valor':    modelo_logit.pvalues
}).drop(index='const').sort_values('p-valor')

print("\n--- Odds Ratios (referencia: Comercio | Micro) ---")
print(or_tabla.round(4).to_string())

# --- Verificación de multicolinealidad (VIF) ---
vif = pd.DataFrame({
    'Variable': X1_const.columns,
    'VIF': [variance_inflation_factor(X1_const.values, i)
            for i in range(X1_const.shape[1])]
}).query("Variable != 'const'").sort_values('VIF', ascending=False)

print("\n--- Factor de Inflación de Varianza (VIF) ---")
print(vif.to_string(index=False))
print("\nNota: VIF > 10 indica multicolinealidad problemática.")

                           Logit Regression Results                           
Dep. Variable:      seguridad_empeoro   No. Observations:                 2021
Model:                          Logit   Df Residuals:                     2005
Method:                           MLE   Df Model:                           15
Date:                Sat, 07 Mar 2026   Pseudo R-squ.:                 0.07082
Time:                        17:39:52   Log-Likelihood:                -1294.1
converged:                       True   LL-Null:                       -1392.7
Covariance Type:                  HC1   LLR p-value:                 7.658e-34
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
const                      0.7671      0.195      3.935      0.000       0.385       1.149
P43_1A                    -0.0702      0.106     -0.665      0.506      -0.277       0.137
P43_

In [51]:
# =============================================================================
# 4B. REGRESIÓN LOGÍSTICA SIN obras_iniciadas
# =============================================================================

X2 = X.drop(columns=['obras_iniciadas'])
X2_const = sm.add_constant(X2)

modelo_logit_2 = Logit(y, X2_const).fit(cov_type='HC1', disp=False)
print(modelo_logit_2.summary())

# --- Odds Ratios con IC 95% ---
or_tabla_2 = pd.DataFrame({
    'Odds Ratio': np.exp(modelo_logit_2.params),
    'IC 2.5%':    np.exp(modelo_logit_2.conf_int()[0]),
    'IC 97.5%':   np.exp(modelo_logit_2.conf_int()[1]),
    'p-valor':    modelo_logit_2.pvalues
}).drop(index='const').sort_values('p-valor')

print("\n--- Odds Ratios (referencia: Comercio | Micro) ---")
print(or_tabla_2.round(4).to_string())

# --- Verificación de multicolinealidad (VIF) ---
vif_2 = pd.DataFrame({
    'Variable': X2_const.columns,
    'VIF': [variance_inflation_factor(X2_const.values, i)
            for i in range(X2_const.shape[1])]
}).query("Variable != 'const'").sort_values('VIF', ascending=False)

print("\n--- Factor de Inflación de Varianza (VIF) ---")
print(vif_2.to_string(index=False))
print("\nNota: VIF > 10 indica multicolinealidad problemática.")

                           Logit Regression Results                           
Dep. Variable:      seguridad_empeoro   No. Observations:                 2021
Model:                          Logit   Df Residuals:                     1988
Method:                           MLE   Df Model:                           32
Date:                Sat, 07 Mar 2026   Pseudo R-squ.:                 0.09154
Time:                        17:39:52   Log-Likelihood:                -1265.2
converged:                       True   LL-Null:                       -1392.7
Covariance Type:                  HC1   LLR p-value:                 1.422e-36
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
const                     -0.0721      0.330     -0.218      0.827      -0.720       0.575
P43_1A                    -0.1108      0.108     -1.024      0.306      -0.323       0.101
P43_